# CIFAR-10 Interpretable CNNs — Extended XAI Analysis


In [32]:
import subprocess, sys
def _pip(pkg, import_name=None):
    try:
        __import__(import_name or pkg.replace('-','_'))
        print(f'  {pkg}')
    except ImportError:
        print(f'  {pkg}...')
        subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
        print(f'   {pkg}')

_pip('grad-cam',       'pytorch_grad_cam')
_pip('scikit-image',   'skimage')
_pip('seaborn')
_pip('pandas')


  grad-cam
  scikit-image
  seaborn
  pandas


In [33]:
import os, copy, time, random, warnings, glob
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as T
import torchvision.models as tv_models
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy.stats import spearmanr
from skimage.metrics import structural_similarity as ssim_metric
from pytorch_grad_cam import GradCAMPlusPlus as LibGradCAMPP
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
plt.style.use('seaborn-v0_8-paper')
for d in ['outputs/gradcam_pp','outputs/dropout','outputs/rand_ssim',
          'outputs/pub_quality','outputs/eval','models']:
    Path(d).mkdir(parents=True, exist_ok=True)
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


PyTorch 2.5.1+cu121 | CUDA: True
GPU: NVIDIA RTX 5000 Ada Generation


##  Setup and Model Loading



In [34]:
CFG = {
    'seed': 42,
    'data': {
        'root':        './data',
        'num_workers': 2,
        'mean': [0.4914, 0.4822, 0.4465],
        'std':  [0.2470, 0.2435, 0.2616],
    },
    'training': {
        'epochs': 100, 'batch_size': 128, 'lr': 0.01,
        'momentum': 0.9, 'weight_decay': 5e-4,
    },
    'models': {'save_dir': './models', 'baseline_cnn': {'dropout': 0.5}},
}
CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']
EXPECTED_ACC = {'baseline_cnn': 87.3, 'resnet18_scratch': 93.3, 'resnet18_pretrained': 95.9}
def set_seed(s=42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [35]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.5):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3,  32, 3, padding=1), nn.BatchNorm2d(32),
            nn.ReLU(inplace=True), nn.MaxPool2d(2))  # 32→16
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),
            nn.ReLU(inplace=True), nn.MaxPool2d(2))  # 16→8
        self.layer3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128),
            nn.ReLU(inplace=True), nn.MaxPool2d(2))  # 8→4
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128*4*4, 256),
            nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(256, num_classes))
    def forward(self, x):
        return self.classifier(self.layer3(self.layer2(self.layer1(x))))


def build_resnet18(pretrained=True, num_classes=10):
    weights = tv_models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model   = tv_models.resnet18(weights=weights)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    tag = 'ImageNet pretrained' if pretrained else 'random init'
    print(f'[model] ResNet18 ({tag})')
    return model
def build_model(model_name, cfg):
    if model_name == 'baseline_cnn':
        return BaselineCNN(dropout=cfg['models']['baseline_cnn']['dropout'])
    elif model_name == 'resnet18_pretrained':
        return build_resnet18(pretrained=True)
    elif model_name == 'resnet18_scratch':
        return build_resnet18(pretrained=False)
    else:
        raise ValueError(f'Unknown model: {model_name}')
param_counts = {'baseline_cnn': 620_000, 'resnet18_scratch': 11_200_000,
                'resnet18_pretrained': 11_200_000}
for name in ['baseline_cnn', 'resnet18_scratch', 'resnet18_pretrained']:
    m = build_model(name, CFG)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{name}: {n:,} params')


baseline_cnn: 620,810 params
[model] ResNet18 (random init)
resnet18_scratch: 11,173,962 params
[model] ResNet18 (ImageNet pretrained)
resnet18_pretrained: 11,173,962 params


In [36]:
MEAN = CFG['data']['mean']
STD  = CFG['data']['std']
def denormalize(tensor, mean=MEAN, std=STD):
    m = torch.tensor(mean).view(3,1,1)
    s = torch.tensor(std).view(3,1,1)
    img = (tensor.cpu().float() * s + m).clamp(0,1)
    return (img.permute(1,2,0).numpy() * 255).astype(np.uint8)
def overlay_heatmap(img_np, cam, alpha=0.4):
    heatmap = (cm.jet(cam)[:,:,:3] * 255).astype(np.uint8)
    blended = ((1-alpha) * img_np.astype(np.float32) / 255
               + alpha  * heatmap.astype(np.float32) / 255)
    return (blended * 255).astype(np.uint8)
def get_target_layer(model, model_name):
    if 'resnet18' in model_name:
        return model.layer4[-1]
    elif 'baseline_cnn' in model_name or 'dropout' in model_name:
        return model.layer3
    raise ValueError(f'Unknown target layer for: {model_name}')
test_transform = T.Compose([T.ToTensor(), T.Normalize(mean=MEAN, std=STD)])


In [37]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer
        self._fmaps = self._grads = None
        self._fwd = target_layer.register_forward_hook(
            lambda m,i,o: setattr(self,'_fmaps',o.detach()))
        self._bwd = target_layer.register_full_backward_hook(
            lambda m,gi,go: setattr(self,'_grads',go[0].detach()))
    def __call__(self, inp, target_class=None, size=(32,32)):
        self.model.eval(); self.model.zero_grad()
        logits = self.model(inp)
        if target_class is None:
            target_class = logits.argmax(1).item()
        logits[0, target_class].backward()
        weights = self._grads.mean(dim=(2,3))
        cam = (weights[:,:,None,None] * self._fmaps).sum(dim=1).squeeze(0)
        cam = F.relu(cam)
        cam = F.interpolate(cam[None,None], size=size,
                            mode='bilinear', align_corners=False).squeeze().cpu().numpy()
        lo, hi = cam.min(), cam.max()
        return (cam - lo) / (hi - lo + 1e-8)
    def remove(self): self._fwd.remove(); self._bwd.remove()
    def __enter__(self): return self
    def __exit__(self, *_): self.remove()
def gradcampp_heatmap(model, target_layer, img_tensor, target_class, device):
    gcampp = LibGradCAMPP(model=model, target_layers=[target_layer])
    inp = img_tensor.unsqueeze(0).to(device)
    cam = gcampp(input_tensor=inp,
                 targets=[ClassifierOutputTarget(target_class)])[0]
    lo, hi = cam.min(), cam.max()
    return (cam - lo) / (hi - lo + 1e-8)



In [38]:
test_dataset = torchvision.datasets.CIFAR10(
    root=CFG['data']['root'], train=False, download=True,
    transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False,
                         num_workers=CFG['data']['num_workers'], pin_memory=True)
print(f'Test set: {len(test_dataset):,} samples | '
      f'Batches: {len(test_loader)} | '
      f'Classes: {len(CLASSES)}')

Files already downloaded and verified
Test set: 10,000 samples | Batches: 40 | Classes: 10


In [39]:
CKPT_PATHS = {
    'baseline_cnn':       ['./models/baseline_cnn.pth',
                           './baseline_cnn.pth',
                           './models/baseline_cnn_best.pth',
                           './models/baseline_cnn_final.pth'],
    'resnet18_scratch':   ['./models/resnet18_scratch.pth',
                           './resnet18_scratch.pth',
                           './models/resnet18_scratch_best.pth',
                           './models/resnet18_scratch_final.pth'],
    'resnet18_pretrained':['./models/resnet18_pretrained.pth',
                           './resnet18_pretrained.pth',
                           './models/resnet18_pretrained_best.pth',
                           './models/resnet18_pretrained_final.pth'],
}
def _load_ckpt(model, model_name, device):
    for p in CKPT_PATHS[model_name]:
        if not os.path.exists(p):
            continue
        state = torch.load(p, map_location=device)
        if isinstance(state, dict) and 'model_state_dict' in state:
            model.load_state_dict(state['model_state_dict'])
        else:
            model.load_state_dict(state)
        print(f'  Loaded {model_name} ← {p}')
        return state
    raise FileNotFoundError(
        f'No checkpoint found for {model_name}. Tried:\n'
        + '\n'.join(f'  {p}' for p in CKPT_PATHS[model_name]))
@torch.no_grad()
def evaluate_accuracy(model, loader, device):
    model.eval()
    correct = total = 0
    all_preds, all_labels = [], []
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        preds = model(imgs).argmax(1)
        correct += (preds == lbls).sum().item()
        total   += lbls.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(lbls.cpu().tolist())
    return 100.0 * correct / total, all_preds, all_labels
LOADED_MODELS    = {}
CKPT_STATES      = {}
MODEL_PREDS      = {}
MODEL_LABELS     = {}
rows = []
for mname in ['baseline_cnn', 'resnet18_scratch', 'resnet18_pretrained']:
    model = build_model(mname, CFG).to(DEVICE)
    state = _load_ckpt(model, mname, DEVICE)
    model.eval()
    LOADED_MODELS[mname] = model
    CKPT_STATES[mname]   = state
    acc, preds, labels = evaluate_accuracy(model, test_loader, DEVICE)
    MODEL_PREDS[mname]  = preds
    MODEL_LABELS[mname] = labels
    exp = EXPECTED_ACC[mname]
    diff = abs(acc - exp)
    ok = diff <= 0.5
    rows.append({'Model': mname, 'Test Acc (%)': round(acc,2),
                 'Expected (%)': exp, 'Diff (pp)': round(diff,2),
                 'Status': '✓ PASS' if ok else '✗ FAIL'})
    assert ok, f'{mname}: {acc:.2f}% vs expected {exp:.1f}% (diff={diff:.2f}pp > 0.5pp)'
print(pd.DataFrame(rows).to_string(index=False))

  Loaded baseline_cnn ← ./models/baseline_cnn_best.pth


C:\Users\User 1\AppData\Local\Temp\ipykernel_65472\2805199062.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(p, map_location=device)


[model] ResNet18 (random init)
  Loaded resnet18_scratch ← ./models/resnet18_scratch_best.pth
[model] ResNet18 (ImageNet pretrained)
  Loaded resnet18_pretrained ← ./models/resnet18_pretrained_best.pth
              Model  Test Acc (%)  Expected (%)  Diff (pp) Status
       baseline_cnn         87.10          87.3       0.20 ✓ PASS
   resnet18_scratch         92.93          93.3       0.37 ✓ PASS
resnet18_pretrained         96.15          95.9       0.25 ✓ PASS


In [40]:
REP_IMAGES = {}
for idx in range(len(test_dataset)):
    img_t, label = test_dataset[idx]
    if label in REP_IMAGES:
        continue
    all_correct = True
    with torch.no_grad():
        for model in LOADED_MODELS.values():
            pred = model(img_t.unsqueeze(0).to(DEVICE)).argmax(1).item()
            if pred != label:
                all_correct = False
                break
    if all_correct:
        REP_IMAGES[label] = img_t
    if len(REP_IMAGES) == 10:
        break
print(f'Representative images selected: {len(REP_IMAGES)} / 10')
for cls_idx, img in sorted(REP_IMAGES.items()):
    print(f'  [{cls_idx}] {CLASSES[cls_idx]}')
assert len(REP_IMAGES) == 10

Representative images selected: 10 / 10
  [0] airplane
  [1] automobile
  [2] bird
  [3] cat
  [4] deer
  [5] dog
  [6] frog
  [7] horse
  [8] ship
  [9] truck



## Grad-CAM++ Comparison



In [41]:
GCAM   = {n: {} for n in LOADED_MODELS}
GCAMPP = {n: {} for n in LOADED_MODELS}
for mname, model in LOADED_MODELS.items():
    model.eval()
    target_layer = get_target_layer(model, mname)
    print(f'Generating heatmaps — {mname}...')
    gcam_ctx = GradCAM(model, target_layer)
    for cls_idx in range(10):
        img_t = REP_IMAGES[cls_idx]
        inp   = img_t.unsqueeze(0).to(DEVICE)
        cam_custom = gcam_ctx(inp, target_class=cls_idx)
        GCAM[mname][cls_idx] = cam_custom
        cam_pp = gradcampp_heatmap(model, target_layer, img_t, cls_idx, DEVICE)
        GCAMPP[mname][cls_idx] = cam_pp
    gcam_ctx.remove()
    print(f'   {mname}: {len(GCAM[mname])} classes done')



Generating heatmaps — baseline_cnn...
   baseline_cnn: 10 classes done
Generating heatmaps — resnet18_scratch...
   resnet18_scratch: 10 classes done
Generating heatmaps — resnet18_pretrained...
   resnet18_pretrained: 10 classes done


In [42]:
spearman_results = {}
ssim_results     = {}
for mname in LOADED_MODELS:
    spearman_results[mname] = []
    ssim_results[mname]     = []
    for cls_idx in range(10):
        cam_gc = GCAM[mname][cls_idx]
        cam_pp = GCAMPP[mname][cls_idx]
        r, _ = spearmanr(cam_gc.flatten(), cam_pp.flatten())
        spearman_results[mname].append(float(r))
        s = ssim_metric(cam_gc, cam_pp, data_range=1.0)
        ssim_results[mname].append(float(s))
summary_rows = []
for mname in LOADED_MODELS:
    r_arr = np.array(spearman_results[mname])
    s_arr = np.array(ssim_results[mname])
    summary_rows.append({
        'Model':           mname,
        'Spearman r (mean)': round(r_arr.mean(),4),
        'Spearman r (std)':  round(r_arr.std(),4),
        'SSIM (mean)':      round(s_arr.mean(),4),
        'SSIM (std)':       round(s_arr.std(),4),
    })

sec2_summary_df = pd.DataFrame(summary_rows)
print(sec2_summary_df.to_string(index=False))


              Model  Spearman r (mean)  Spearman r (std)  SSIM (mean)  SSIM (std)
       baseline_cnn             0.4850            0.3743       0.4150      0.2048
   resnet18_scratch             0.9766            0.0517       0.9724      0.0487
resnet18_pretrained             0.9934            0.0114       0.9930      0.0124


In [43]:
SELECTED_5 = [4, 3, 5, 7, 2]
for mname, model in LOADED_MODELS.items():
    fig, axes = plt.subplots(5, 4, figsize=(16, 12))
    fig.suptitle(f'Grad-CAM vs Grad-CAM++ — {mname}', fontsize=14, fontweight='bold', y=1.01)
    col_titles = ['Original', 'Grad-CAM', 'Grad-CAM++', '|Difference|']
    for j, ct in enumerate(col_titles):
        axes[0, j].set_title(ct, fontsize=11, fontweight='bold')
    for row, cls_idx in enumerate(SELECTED_5):
        img_t  = REP_IMAGES[cls_idx]
        img_np = denormalize(img_t)
        cam_gc = GCAM[mname][cls_idx]
        cam_pp = GCAMPP[mname][cls_idx]
        diff   = np.abs(cam_gc - cam_pp)
        axes[row, 0].imshow(img_np)
        axes[row, 1].imshow(overlay_heatmap(img_np, cam_gc, alpha=0.4))
        axes[row, 2].imshow(overlay_heatmap(img_np, cam_pp, alpha=0.4))
        im = axes[row, 3].imshow(diff, cmap='hot', vmin=0, vmax=1)
        for j in range(4):
            axes[row, j].axis('off')
        axes[row, 0].set_ylabel(CLASSES[cls_idx], fontsize=10,
                                rotation=0, labelpad=50, va='center')
        r_val = spearman_results[mname][cls_idx]
        s_val = ssim_results[mname][cls_idx]
        axes[row, 1].set_xlabel(f'r={r_val:.3f}', fontsize=8)
        axes[row, 2].set_xlabel(f'SSIM={s_val:.3f}', fontsize=8)
    plt.colorbar(im, ax=axes[:, 3], shrink=0.6, label='|CAM - CAM++|')
    plt.tight_layout()
    save_p = f'outputs/gradcam_pp/gradcam_pp_comparison_{mname}.png'
    plt.savefig(save_p, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: {save_p}')


C:\Users\User 1\AppData\Local\Temp\ipykernel_65472\221306237.py:27: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Saved: outputs/gradcam_pp/gradcam_pp_comparison_baseline_cnn.png
Saved: outputs/gradcam_pp/gradcam_pp_comparison_resnet18_scratch.png
Saved: outputs/gradcam_pp/gradcam_pp_comparison_resnet18_pretrained.png


In [44]:
DOG_CLS = 5
dog_img  = REP_IMAGES[DOG_CLS]
dog_np   = denormalize(dog_img)
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle('Grad-CAM vs Grad-CAM++ — Dog Class Across All Models',
             fontsize=14, fontweight='bold')
for row, mname in enumerate(['baseline_cnn','resnet18_scratch','resnet18_pretrained']):
    cam_gc = GCAM[mname][DOG_CLS]
    cam_pp = GCAMPP[mname][DOG_CLS]
    r_val  = spearman_results[mname][DOG_CLS]
    s_val  = ssim_results[mname][DOG_CLS]
    axes[row, 0].imshow(dog_np)
    axes[row, 0].set_title(f'{mname}\nOriginal', fontsize=9)
    axes[row, 1].imshow(overlay_heatmap(dog_np, cam_gc, alpha=0.4))
    axes[row, 1].set_title(f'Grad-CAM\nr={r_val:.3f}', fontsize=9)
    axes[row, 2].imshow(overlay_heatmap(dog_np, cam_pp, alpha=0.4))
    axes[row, 2].set_title(f'Grad-CAM++\nSSIM={s_val:.3f}', fontsize=9)
    for j in range(3):
        axes[row, j].axis('off')
plt.tight_layout()
save_p = 'outputs/gradcam_pp/gradcam_vs_gradcampp_dog_summary.png'
plt.savefig(save_p, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {save_p}')


Saved: outputs/gradcam_pp/gradcam_vs_gradcampp_dog_summary.png


In [45]:

model_labels = {'baseline_cnn': 'BaselineCNN',
                'resnet18_scratch': 'ResNet18-scratch',
                'resnet18_pretrained': 'ResNet18-pretrained'}
for mname in LOADED_MODELS:
    r_arr = np.array(spearman_results[mname])
    s_arr = np.array(ssim_results[mname])
    lbl   = model_labels[mname]
    print(f'{lbl} & {r_arr.mean():.4f} & $\\pm${r_arr.std():.4f} '
          f'& {s_arr.mean():.4f} & $\\pm${s_arr.std():.4f} \\\\')


BaselineCNN & 0.4850 & $\pm$0.3743 & 0.4150 & $\pm$0.2048 \\
ResNet18-scratch & 0.9766 & $\pm$0.0517 & 0.9724 & $\pm$0.0487 \\
ResNet18-pretrained & 0.9934 & $\pm$0.0114 & 0.9930 & $\pm$0.0124 \\


## Dropout Ablation Study (BaselineCNN)



In [46]:
def get_train_val_loaders(cfg):
    train_transform = T.Compose([
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(p=0.5),
        T.ToTensor(),
        T.Normalize(mean=cfg['data']['mean'], std=cfg['data']['std']),
    ])
    train_set = torchvision.datasets.CIFAR10(
        root=cfg['data']['root'], train=True, download=True,
        transform=train_transform)
    val_set = torchvision.datasets.CIFAR10(
        root=cfg['data']['root'], train=False, download=True,
        transform=test_transform)
    bs = cfg['training']['batch_size']
    nw = cfg['data']['num_workers']
    return (DataLoader(train_set, batch_size=bs, shuffle=True,  num_workers=nw, pin_memory=True),
            DataLoader(val_set,   batch_size=bs, shuffle=False, num_workers=nw, pin_memory=True))
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = total_correct = total_n = 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), lbls)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * imgs.size(0)
        total_correct += (model(imgs).detach().argmax(1) == lbls).sum().item()
        total_n       += imgs.size(0)
    return total_loss / total_n, 100.0 * total_correct / total_n
@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_n = 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        logits = model(imgs)
        total_loss    += criterion(logits, lbls).item() * imgs.size(0)
        total_correct += (logits.argmax(1) == lbls).sum().item()
        total_n       += imgs.size(0)
    return total_loss / total_n, 100.0 * total_correct / total_n

In [47]:
DROPOUT_RATES      = [0.0, 0.2, 0.4, 0.6, 0.8]
DROPOUT_MODELS     = {}  # rate -> trained model
DROPOUT_CURVES     = {}  # rate -> {'train_loss','val_loss','train_acc','val_acc'}
DROPOUT_FINAL_ACC  = {}  # rate -> final test accuracy
criterion  = nn.CrossEntropyLoss()
train_loader, val_loader = get_train_val_loaders(CFG)
for dr in DROPOUT_RATES:
    set_seed(42)  # Same seed for every rate
    dr_str = str(dr).replace('.', '')
    ckpt_p = f'models/baseline_dropout_{dr_str}.pth'
    if os.path.exists(ckpt_p):
        print(f'[dr={dr}] Checkpoint found — loading {ckpt_p}')
        model = BaselineCNN(dropout=dr).to(DEVICE)
        state = torch.load(ckpt_p, map_location=DEVICE)
        model.load_state_dict(state['model_state_dict'])
        DROPOUT_MODELS[dr]    = model
        DROPOUT_CURVES[dr]    = {k: state[k] for k in
                                 ['train_loss','val_loss','train_acc','val_acc']}
        DROPOUT_FINAL_ACC[dr] = state.get('test_acc', 0.0)
        continue
    print(f'\n[dr={dr}] Training BaselineCNN from scratch (100 epochs)...')
    model     = BaselineCNN(dropout=dr).to(DEVICE)
    optimizer = optim.SGD(model.parameters(), lr=CFG['training']['lr'],
                          momentum=CFG['training']['momentum'],
                          weight_decay=CFG['training']['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,
                    T_max=CFG['training']['epochs'])
    curves = {'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[]}
    best_val_acc = 0.0
    t0 = time.time()
    for ep in range(1, CFG['training']['epochs'] + 1):
        tl, ta = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        vl, va = eval_epoch(model, val_loader, criterion, DEVICE)
        scheduler.step()
        curves['train_loss'].append(tl); curves['val_loss'].append(vl)
        curves['train_acc'].append(ta);  curves['val_acc'].append(va)
        if va > best_val_acc:
            best_val_acc = va
        if ep % 20 == 0 or ep == 1:
            elapsed = time.time() - t0
            print(f'  ep {ep:3d}/100 | tl={tl:.4f} ta={ta:.1f}% '
                  f'| vl={vl:.4f} va={va:.1f}% | {elapsed:.0f}s')
    test_acc, _, _ = evaluate_accuracy(model, test_loader, DEVICE)
    print(f'  [dr={dr}] Final test acc: {test_acc:.2f}% | best val: {best_val_acc:.2f}%')
    torch.save({'model_state_dict': model.state_dict(),
                'test_acc': test_acc, 'best_val_acc': best_val_acc,
                **curves}, ckpt_p)
    DROPOUT_MODELS[dr]    = model
    DROPOUT_CURVES[dr]    = curves
    DROPOUT_FINAL_ACC[dr] = test_acc
    print(f'  Saved: {ckpt_p}')
for dr in DROPOUT_RATES:
    print(f'  dr={dr}: test_acc={DROPOUT_FINAL_ACC[dr]:.2f}%')


Files already downloaded and verified
Files already downloaded and verified
[dr=0.0] Checkpoint found — loading models/baseline_dropout_00.pth
[dr=0.2] Checkpoint found — loading models/baseline_dropout_02.pth
[dr=0.4] Checkpoint found — loading models/baseline_dropout_04.pth
[dr=0.6] Checkpoint found — loading models/baseline_dropout_06.pth
[dr=0.8] Checkpoint found — loading models/baseline_dropout_08.pth
  dr=0.0: test_acc=88.06%
  dr=0.2: test_acc=88.02%
  dr=0.4: test_acc=87.82%
  dr=0.6: test_acc=86.07%
  dr=0.8: test_acc=86.60%


C:\Users\User 1\AppData\Local\Temp\ipykernel_65472\3204293580.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt_p, map_location=DEVICE)


In [48]:
rates = DROPOUT_RATES
accs  = [DROPOUT_FINAL_ACC[r] for r in rates]
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rates, accs, 'o-', color='#2563EB', lw=2, ms=8, label='Ablated models')

orig_acc = EXPECTED_ACC['baseline_cnn']
ax.axhline(orig_acc, color='#DC2626', ls='--', lw=1.5, label=f'Original dropout=0.5 ({orig_acc:.1f}%)')
ax.axvline(0.5, color='#DC2626', ls=':', lw=1, alpha=0.6)
ax.set_xlabel('Dropout Rate', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('BaselineCNN: Test Accuracy vs Dropout Rate', fontsize=13)
ax.set_xticks(rates)
ax.legend()
ax.grid(alpha=0.3)
for r, a in zip(rates, accs):
    ax.annotate(f'{a:.1f}%', (r, a), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('outputs/dropout/dropout_accuracy_curve.png', dpi=150, bbox_inches='tight')
plt.close()


In [49]:
colors  = ['#3B82F6','#10B981','#F59E0B','#EF4444','#8B5CF6']
fig, ax = plt.subplots(figsize=(12, 5))
for dr, color in zip(DROPOUT_RATES, colors):
    curves = DROPOUT_CURVES[dr]
    epochs = range(1, len(curves['train_loss']) + 1)
    ax.plot(epochs, curves['train_loss'], color=color, lw=1.5,
            ls='-',  label=f'dr={dr} train')
    ax.plot(epochs, curves['val_loss'],   color=color, lw=1.5,
            ls='--', label=f'dr={dr} val')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('BaselineCNN: Train (solid) / Val (dashed) Loss — All Dropout Rates')
ax.legend(ncol=5, fontsize=7, loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/dropout/dropout_training_curves.png', dpi=150, bbox_inches='tight')
plt.close()

In [50]:
DROPOUT_GCAM = {}
for dr, model in DROPOUT_MODELS.items():
    model.eval()
    DROPOUT_GCAM[dr] = {}
    tl = model.layer3
    gcam_ctx = GradCAM(model, tl)
    for cls_idx in range(10):
        img_t = REP_IMAGES[cls_idx]
        inp   = img_t.unsqueeze(0).to(DEVICE)
        DROPOUT_GCAM[dr][cls_idx] = gcam_ctx(inp, target_class=cls_idx)
    gcam_ctx.remove()
    print(f'  dr={dr}: heatmaps generated')
orig_model = LOADED_MODELS['baseline_cnn']
orig_model.eval()
ORIG_GCAM = {}
gcam_ctx = GradCAM(orig_model, orig_model.layer3)
for cls_idx in range(10):
    img_t = REP_IMAGES[cls_idx]
    ORIG_GCAM[cls_idx] = gcam_ctx(img_t.unsqueeze(0).to(DEVICE), target_class=cls_idx)
gcam_ctx.remove()

  dr=0.0: heatmaps generated
  dr=0.2: heatmaps generated
  dr=0.4: heatmaps generated
  dr=0.6: heatmaps generated
  dr=0.8: heatmaps generated


In [51]:
GRID_CLASSES = [0, 1, 3, 5, 7]  # airplane, automobile, cat, dog, horse
GRID_CLS_NAMES = [CLASSES[i] for i in GRID_CLASSES]
n_rates = len(DROPOUT_RATES)
n_cls   = len(GRID_CLASSES)
fig, axes = plt.subplots(n_rates, n_cls * 2, figsize=(14, 10))
fig.suptitle('Grad-CAM Grid: Dropout Rate × Class', fontsize=13, fontweight='bold')
for j, cname in enumerate(GRID_CLS_NAMES):
    axes[0, j*2].set_title(cname, fontsize=9, fontweight='bold')
for row, dr in enumerate(DROPOUT_RATES):
    axes[row, 0].set_ylabel(f'dr={dr}', fontsize=9, rotation=0, labelpad=45, va='center')
    for col, cls_idx in enumerate(GRID_CLASSES):
        img_np = denormalize(REP_IMAGES[cls_idx])
        cam    = DROPOUT_GCAM[dr][cls_idx]
        axes[row, col*2  ].imshow(img_np)
        axes[row, col*2+1].imshow(overlay_heatmap(img_np, cam, alpha=0.4))
        for j in range(2):
            axes[row, col*2+j].axis('off')
plt.tight_layout()
plt.savefig('outputs/dropout/dropout_gradcam_grid.png', dpi=150, bbox_inches='tight')
plt.close()

In [52]:
dropout_ssim_means = []
dropout_ssim_stds  = []
for dr in DROPOUT_RATES:
    ssim_vals = []
    for cls_idx in range(10):
        s = ssim_metric(ORIG_GCAM[cls_idx], DROPOUT_GCAM[dr][cls_idx], data_range=1.0)
        ssim_vals.append(s)
    dropout_ssim_means.append(np.mean(ssim_vals))
    dropout_ssim_stds.append(np.std(ssim_vals))
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([str(r) for r in DROPOUT_RATES], dropout_ssim_means,
             yerr=dropout_ssim_stds, color='#2563EB', alpha=0.8,
             capsize=5, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Dropout Rate', fontsize=12)
ax.set_ylabel('Mean SSIM vs dropout=0.5', fontsize=12)
ax.set_title('Grad-CAM Stability: SSIM vs Original (dropout=0.5)', fontsize=13)
ax.set_ylim(0, 1.05)
for bar, m in zip(bars, dropout_ssim_means):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
            f'{m:.3f}', ha='center', va='bottom', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/dropout/dropout_ssim_comparison.png', dpi=150, bbox_inches='tight')
plt.close()


In [53]:
def interpret_dropout(dr, acc, ssim_val):
    if dr == 0.0:     return
    elif dr <= 0.2:   return
    elif dr <= 0.4:   return
    elif dr <= 0.6:   return
    else:             return
for dr, m_ssim in zip(DROPOUT_RATES, dropout_ssim_means):
    acc = DROPOUT_FINAL_ACC[dr]
    interp = interpret_dropout(dr, acc, m_ssim)
    print(f'{dr} & {acc:.2f} & {m_ssim:.4f} & {interp} \\\\')
print(f'0.5 (orig) & {orig_acc:.2f} & 1.0000 & Reference model \\\\')



0.0 & 88.06 & 0.4882 & None \\
0.2 & 88.02 & 0.4182 & None \\
0.4 & 87.82 & 0.4447 & None \\
0.6 & 86.07 & 0.4555 & None \\
0.8 & 86.60 & 0.4530 & None \\
0.5 (orig) & 87.30 & 1.0000 & Reference model \\


## Model Randomization SSIM Quantification



In [54]:
def get_layer_groups(model, model_name):
    if 'resnet18' in model_name:
        return [
            ('fc',     model.fc),
            ('layer4', model.layer4),
            ('layer3', model.layer3),
            ('layer2', model.layer2),
            ('layer1', model.layer1),
        ]
    else:
        return [
            ('classifier', model.classifier),
            ('layer3',     model.layer3),
            ('layer2',     model.layer2),
            ('layer1',     model.layer1),
        ]

def randomize_layer_group(layer_group):
    for mod in (layer_group.modules() if hasattr(layer_group, 'modules')
                else [layer_group]):
        for p in mod.parameters(recurse=False):
            nn.init.normal_(p)
def compute_rand_ssim(model, model_name, rep_images, device, n_stages=5):
    model_copy   = copy.deepcopy(model)
    model_copy.eval()
    layer_groups = get_layer_groups(model_copy, model_name)
    target_layer = get_target_layer(model_copy, model_name)
    orig_cams = {}
    gcam_ctx  = GradCAM(model_copy, target_layer)
    for cls_idx in range(10):
        img_t = rep_images[cls_idx]
        orig_cams[cls_idx] = gcam_ctx(img_t.unsqueeze(0).to(device),
                                       target_class=cls_idx)
    gcam_ctx.remove()
    ssim_means, ssim_stds = [], []
    n_stages = min(n_stages, len(layer_groups))

    for stage in range(1, n_stages + 1):
        _, layer_g = layer_groups[stage - 1]
        randomize_layer_group(layer_g)
        target_layer = get_target_layer(model_copy, model_name)
        gcam_ctx = GradCAM(model_copy, target_layer)
        stage_ssims = []
        for cls_idx in range(10):
            img_t    = rep_images[cls_idx]
            rand_cam = gcam_ctx(img_t.unsqueeze(0).to(device), target_class=cls_idx)
            s        = ssim_metric(orig_cams[cls_idx], rand_cam, data_range=1.0)
            stage_ssims.append(s)
        gcam_ctx.remove()
        ssim_means.append(float(np.mean(stage_ssims)))
        ssim_stds.append(float(np.std(stage_ssims)))
        print(f'  Stage {stage}: mean SSIM={ssim_means[-1]:.4f} ± {ssim_stds[-1]:.4f}')
    del model_copy
    return np.array(ssim_means), np.array(ssim_stds)
RAND_SSIM_MEANS = {}
RAND_SSIM_STDS  = {}

for mname, model in LOADED_MODELS.items():
    print(f'\nRandomization SSIM — {mname}')
    means, stds = compute_rand_ssim(model, mname, REP_IMAGES, DEVICE, n_stages=5)
    RAND_SSIM_MEANS[mname] = means
    RAND_SSIM_STDS[mname]  = stds


Randomization SSIM — baseline_cnn
  Stage 1: mean SSIM=0.0606 ± 0.1429
  Stage 2: mean SSIM=0.0601 ± 0.1299
  Stage 3: mean SSIM=0.0577 ± 0.1864
  Stage 4: mean SSIM=0.0606 ± 0.1403

Randomization SSIM — resnet18_scratch
  Stage 1: mean SSIM=0.0953 ± 0.2688
  Stage 2: mean SSIM=-0.0047 ± 0.1222
  Stage 3: mean SSIM=0.0213 ± 0.1149
  Stage 4: mean SSIM=-0.0645 ± 0.0738
  Stage 5: mean SSIM=-0.0194 ± 0.0740

Randomization SSIM — resnet18_pretrained
  Stage 1: mean SSIM=0.2953 ± 0.3635
  Stage 2: mean SSIM=0.2377 ± 0.2849
  Stage 3: mean SSIM=0.0276 ± 0.2445
  Stage 4: mean SSIM=0.0921 ± 0.2397
  Stage 5: mean SSIM=nan ± nan


In [55]:
for mname in LOADED_MODELS:
    means = RAND_SSIM_MEANS[mname]
    stds  = RAND_SSIM_STDS[mname]
    stages = np.arange(1, len(means) + 1)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot([0] + list(stages), [1.0] + list(means),
            'o-', color='#2563EB', lw=2, ms=8)
    ax.fill_between([0]+list(stages),
                    [1.0]+list(means - stds),
                    [1.0]+list(means + stds),
                    alpha=0.2, color='#2563EB', label='±1 std')
    ax.axhline(0.5, color='#DC2626', ls='--', lw=1, label='SSIM=0.5 threshold')
    ax.set_xlabel('Cascade Stage (0=original, 5=fully randomized)', fontsize=11)
    ax.set_ylabel('Mean SSIM vs Stage-0', fontsize=11)
    ax.set_title(f'Model Randomization SSIM Decay — {mname}', fontsize=12)
    ax.set_xticks([0]+list(stages))
    ax.set_xticklabels(['Original']+[f'Stage {s}' for s in stages], rotation=15)
    ax.set_ylim(0, 1.1)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    save_p = f'outputs/rand_ssim/model_rand_ssim_{mname}.png'
    plt.savefig(save_p, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: {save_p}')


Saved: outputs/rand_ssim/model_rand_ssim_baseline_cnn.png
Saved: outputs/rand_ssim/model_rand_ssim_resnet18_scratch.png
Saved: outputs/rand_ssim/model_rand_ssim_resnet18_pretrained.png


In [56]:
model_colors = {'baseline_cnn': '#3B82F6',
                'resnet18_scratch': '#10B981',
                'resnet18_pretrained': '#F59E0B'}
model_labels_nice = {'baseline_cnn': 'BaselineCNN',
                     'resnet18_scratch': 'ResNet18-scratch',
                     'resnet18_pretrained': 'ResNet18-pretrained'}
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
fig.suptitle('Model Randomization SSIM Decay — All Models', fontsize=14, fontweight='bold')
for ax, mname in zip(axes, ['baseline_cnn','resnet18_scratch','resnet18_pretrained']):
    means = RAND_SSIM_MEANS[mname]
    stds  = RAND_SSIM_STDS[mname]
    stages = np.arange(1, len(means) + 1)
    color = model_colors[mname]
    x = [0] + list(stages)
    y = [1.0] + list(means)
    y_lo = [1.0] + list(means - stds)
    y_hi = [1.0] + list(means + stds)
    ax.plot(x, y, 'o-', color=color, lw=2, ms=8)
    ax.fill_between(x, y_lo, y_hi, alpha=0.2, color=color)
    ax.axhline(0.5, color='#DC2626', ls='--', lw=1, alpha=0.7)
    ax.set_title(model_labels_nice[mname], fontsize=11)
    ax.set_xlabel('Cascade Stage')
    ax.set_xticks(x)
    ax.set_xticklabels(['Orig']+[f'S{s}' for s in stages])
    ax.set_ylim(0, 1.1)
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Mean SSIM vs Stage-0', fontsize=11)
plt.tight_layout()
plt.savefig('outputs/rand_ssim/model_rand_ssim_all_models.png', dpi=150, bbox_inches='tight')
plt.close()


In [57]:
def rand_verdict(means):
    if means[0] < 0.7:   return 'PASSES — rapid decay'
    elif means[-1] < 0.4: return 'PASSES — decay at depth'
    elif means[-1] > 0.7: return 'FAILS  — low sensitivity'
    else:                 return 'INCONCLUSIVE'
for mname in LOADED_MODELS:
    means = RAND_SSIM_MEANS[mname]
    # Pad to 5 stages if fewer computed
    m5 = list(means) + [float('nan')]*(5-len(means))
    verdict = rand_verdict(means)
    label = model_labels_nice[mname]
    vals = ' & '.join(f'{v:.4f}' if not np.isnan(v) else '--' for v in m5)
    print(f'{label} & {vals} & {verdict} \\\\')

BaselineCNN & 0.0606 & 0.0601 & 0.0577 & 0.0606 & -- & PASSES — rapid decay \\
ResNet18-scratch & 0.0953 & -0.0047 & 0.0213 & -0.0645 & -0.0194 & PASSES — rapid decay \\
ResNet18-pretrained & 0.2953 & 0.2377 & 0.0276 & 0.0921 & -- & PASSES — rapid decay \\


##  Publication-Quality Figure Regeneration



In [58]:
for mname, model in LOADED_MODELS.items():
    n_rows, n_cols = 10, 2
    cell_size = 2.5
    spacing   = 0.3
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * cell_size, n_rows * cell_size),
                             gridspec_kw={'hspace': spacing, 'wspace': spacing})
    fig.suptitle(f'Grad-CAM Heatmaps — {model_labels_nice.get(mname, mname)}',
                 fontsize=13, fontweight='bold', y=1.01)
    axes[0, 0].set_title('Original',  fontsize=10)
    axes[0, 1].set_title('Grad-CAM',  fontsize=10)
    for cls_idx in range(10):
        img_t  = REP_IMAGES[cls_idx]
        img_np = denormalize(img_t)
        cam    = GCAM[mname][cls_idx]
        overlay = overlay_heatmap(img_np, cam, alpha=0.4)
        axes[cls_idx, 0].imshow(img_np)
        axes[cls_idx, 1].imshow(overlay)
        axes[cls_idx, 0].set_ylabel(CLASSES[cls_idx], fontsize=9,
                                    rotation=0, labelpad=45, va='center')
        for j in range(2):
            axes[cls_idx, j].set_xticks([])
            axes[cls_idx, j].set_yticks([])
            for spine in axes[cls_idx, j].spines.values():
                spine.set_visible(False)
    save_p = f'outputs/pub_quality/gradcam_grid_{mname}_final.png'
    plt.savefig(save_p, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: {save_p}')


Saved: outputs/pub_quality/gradcam_grid_baseline_cnn_final.png
Saved: outputs/pub_quality/gradcam_grid_resnet18_scratch_final.png
Saved: outputs/pub_quality/gradcam_grid_resnet18_pretrained_final.png


In [59]:
for mname in LOADED_MODELS:
    preds  = MODEL_PREDS[mname]
    labels = MODEL_LABELS[mname]
    cm_mat = np.zeros((10, 10), dtype=int)
    for p, l in zip(preds, labels):
        cm_mat[l, p] += 1
    per_class_acc = cm_mat.diagonal() / cm_mat.sum(axis=1)
    annot_mat = np.empty_like(cm_mat, dtype=object)
    for i in range(10):
        for j in range(10):
            if i == j:
                annot_mat[i, j] = f'{cm_mat[i,j]}\n({per_class_acc[i]*100:.1f}%)'
            else:
                annot_mat[i, j] = str(cm_mat[i, j]) if cm_mat[i,j] > 0 else ''
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cm_mat, annot=annot_mat, fmt='', cmap='Blues',
                ax=ax, linewidths=0.5, linecolor='gray',
                xticklabels=CLASSES, yticklabels=CLASSES)
    ax.set_xlabel('Predicted Class', fontsize=13)
    ax.set_ylabel('True Class', fontsize=13)
    overall_acc = 100.0 * sum(p==l for p,l in zip(preds,labels)) / len(labels)
    ax.set_title(f'{model_labels_nice.get(mname, mname)} — Confusion Matrix '
                 f'(Overall Acc: {overall_acc:.2f}%)', fontsize=13, fontweight='bold')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=10)
    plt.setp(ax.get_yticklabels(), rotation=0,  fontsize=10)
    plt.tight_layout()
    save_p = f'outputs/pub_quality/confusion_matrix_{mname}_final.png'
    plt.savefig(save_p, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: {save_p}')

Saved: outputs/pub_quality/confusion_matrix_baseline_cnn_final.png
Saved: outputs/pub_quality/confusion_matrix_resnet18_scratch_final.png
Saved: outputs/pub_quality/confusion_matrix_resnet18_pretrained_final.png


In [60]:
for mname in LOADED_MODELS:
    state = CKPT_STATES[mname]
    train_losses = state.get('train_losses', None)
    val_losses   = state.get('val_losses',   None)
    train_accs   = state.get('train_accs',   None)
    val_accs     = state.get('val_accs',     None)
    if train_losses is None or len(train_losses) == 0:
        print(f'  [{mname}] No curve data in checkpoint — skipping training curves.')
        continue
    epochs  = range(1, len(train_losses) + 1)
    best_va = max(val_accs)
    best_ep = val_accs.index(best_va) + 1
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{model_labels_nice.get(mname, mname)} — Training Curves',
                 fontsize=13, fontweight='bold')
    ax1.plot(epochs, train_losses, color='#2563EB', lw=1.5, label='Train Loss')
    ax1.plot(epochs, val_losses,   color='#DC2626', lw=1.5, ls='--', label='Val Loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy Loss')
    ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(epochs, train_accs, color='#2563EB', lw=1.5, label='Train Acc')
    ax2.plot(epochs, val_accs,   color='#DC2626', lw=1.5, ls='--', label='Val Acc')
    ax2.annotate(f'Best: {best_va:.1f}%\n(ep {best_ep})',
                 xy=(best_ep, best_va),
                 xytext=(best_ep + max(len(epochs)//10, 5), best_va - 5),
                 arrowprops=dict(arrowstyle='->', color='black', lw=1),
                 fontsize=9, color='black')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout()
    save_p = f'outputs/pub_quality/training_curves_{mname}_final.png'
    plt.savefig(save_p, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: {save_p}')

  [baseline_cnn] No curve data in checkpoint — skipping training curves.
  [resnet18_scratch] No curve data in checkpoint — skipping training curves.
  [resnet18_pretrained] No curve data in checkpoint — skipping training curves.


## Summary


In [61]:
print('\nCIFAR-10 XAI Extended Analysis — Results Summary')
print('\n§1  TEST ACCURACY')
print(f'{"Model":<25} {"Acc (%)":>10} {"Expected":>10} {"Diff (pp)":>12}')
print('-' * 60)
for mname in LOADED_MODELS:
    preds  = MODEL_PREDS[mname]
    acc    = sum(p == l for p, l in zip(preds, MODEL_LABELS[mname])) / len(preds) * 100
    exp  = EXPECTED_ACC[mname]
    diff = abs(acc - exp)
    print(f'{mname:<25} {acc:>10.2f} {exp:>10.1f} {diff:>12.2f}')
print('\n§2  GRAD-CAM vs GRAD-CAM++ (Spearman r | SSIM)')
print(f'{"Model":<25} {"r (mean±std)":>18} {"SSIM (mean±std)":>18}')
for mname in LOADED_MODELS:
    r_arr = np.array(spearman_results[mname])
    s_arr = np.array(ssim_results[mname])
    print(f'{mname:<25} {r_arr.mean():.4f}±{r_arr.std():.4f} '
          f'{s_arr.mean():>12.4f}±{s_arr.std():.4f}')
print('\n§3  DROPOUT ABLATION (BaselineCNN)')
print(f'{"Dropout":>10} {"Test Acc (%)":>14} {"SSIM vs 0.5":>14}')
for dr, m_ssim in zip(DROPOUT_RATES, dropout_ssim_means):
    acc = DROPOUT_FINAL_ACC[dr]
    print(f'{dr:>10.1f} {acc:>14.2f} {m_ssim:>14.4f}')
print(f'{"0.5 (orig)":>10} {orig_acc:>14.2f} {1.0:>14.4f}')
print('\n§4  MODEL RANDOMIZATION SSIM (stage 1 → 5)')
print(f'{"Model":<25}', end='')
for s in range(1, 6): print(f'  Stage {s}', end='')
for mname in LOADED_MODELS:
    means = RAND_SSIM_MEANS[mname]
    m5 = list(means) + [float('nan')]*(5-len(means))
    vals = '  '.join(f'{v:.4f}' if not np.isnan(v) else '  --  ' for v in m5)
    print(f'{mname:<25}  {vals}')
print('\n§5  OUTPUT FILES')
for pattern in ['outputs/gradcam_pp/*.png', 'outputs/dropout/*.png',
                'outputs/rand_ssim/*.png',  'outputs/pub_quality/*.png']:
    for f in sorted(glob.glob(pattern)):
        print(f'  {f}')
print('All sections complete. Fill values into report / LOG.md.')




CIFAR-10 XAI Extended Analysis — Results Summary

§1  TEST ACCURACY
Model                        Acc (%)   Expected    Diff (pp)
------------------------------------------------------------
baseline_cnn                   87.10       87.3         0.20
resnet18_scratch               92.93       93.3         0.37
resnet18_pretrained            96.15       95.9         0.25

§2  GRAD-CAM vs GRAD-CAM++ (Spearman r | SSIM)
Model                           r (mean±std)    SSIM (mean±std)
baseline_cnn              0.4850±0.3743       0.4150±0.2048
resnet18_scratch          0.9766±0.0517       0.9724±0.0487
resnet18_pretrained       0.9934±0.0114       0.9930±0.0124

§3  DROPOUT ABLATION (BaselineCNN)
   Dropout   Test Acc (%)    SSIM vs 0.5
       0.0          88.06         0.4882
       0.2          88.02         0.4182
       0.4          87.82         0.4447
       0.6          86.07         0.4555
       0.8          86.60         0.4530
0.5 (orig)          87.30         1.0000

§4  MODEL 